# HoloViews 的基本用法

HoloViews 是架在别的绘图库之上的可视化与分析库：给数据加上“这是哪种图”“哪一列是哪个维度”这样的注记，它就把图做出来，真正的绘制由 matplotlib 或 Bokeh 完成。

在 Notebook 里使用时，先用 `extension()` 选好绘制用的库，用 Bokeh 就是下面这样。

```python
hv.extension('bokeh')
```

这一章说明用 HoloViews 做可视化应用的办法，同时介绍 Param 与 Panel：把这两个与 HoloViews 组合起来，交互式的应用很容易做。

- `Element` 与 `Dimension`
- 从整齐数据做出 `Dataset`
- 容器(`HoloMap`、`NdLayout`、`NdOverlay`、`Layout`、`Overlay`)
- 图的选项(`opts`)

In [1]:
import pprint
import holoviews as hv
from holoviews import streams
import numpy as np
import pandas as pd
from holoviews import opts
import panel as pn

hv.extension("bokeh", inline=False, logo=False)
pn.extension()

## `Element` 对象

HoloViews 里用 `Element` 的派生类给数据加注记。下面是 `Element` 的派生树：曲线的 `Curve` 从 `Chart` 派生，而 `Chart` 从 `Dataset` 与 `Element2D` 派生。

In [2]:
from helper.python import print_subclasses

text = print_subclasses(hv.Element, return_str=True)
pn.widgets.TextAreaInput(
    value=text, width=500, height=300, styles={"font-family": "monospace"}
)

TextAreaInput(height=300, sizing_mode='fixed', styles={'font-family': '...}, value='└──Element\n  ..., width=500)

`Element` 能接收各种形状的数据：下面的程序做出元组、`DataFrame`、`ndarray`、字典 4 种，分别用 `Curve` 包起来；为了便于区分改了 `label`，再查 `data` 的类型。`c1` 与 `c2` 变成 `DataFrame`、`c3` 是 `ndarray`、`c4` 是 `OrderedDict`，元组则被转换成 `DataFrame`。

In [3]:
x = np.linspace(0, 4 * np.pi, 200)
y = np.sin(x)

tuple_data = (x, y)
df_data = pd.DataFrame(dict(x=x, y=y))
arr_data = np.c_[x, y]
dict_data = dict(x=x, y=y)

c1 = hv.Curve(tuple_data, label="c1")
c2 = hv.Curve(df_data, label="c2")
c3 = hv.Curve(arr_data, label="c3")
c4 = hv.Curve(dict_data, label="c4")

curves = [c1, c2, c3, c4]
print(" ".join(f"{c.label}: {type(c.data).__name__}" for c in curves))

c1: DataFrame c2: DataFrame c3: ndarray c4: OrderedDict


类型不同，数据存放的方式(接口)也不同，用 `interface` 属性就能知道。

In [4]:
print(" ".join(f"{c.label}: {c.interface.__name__}" for c in curves))

c1: PandasInterface c2: PandasInterface c3: ArrayInterface c4: DictInterface


`c3` 是数组，不过用 `dframe()` 能转换成 `DataFrame`。

In [5]:
print(c3.dframe().head())

          x         y
0  0.000000  0.000000
1  0.063148  0.063106
2  0.126295  0.125960
3  0.189443  0.188312
4  0.252590  0.249913


HoloViews 的类从 `param.Parameterized` 派生，所以用 `.param.values()` 能看到全部属性，其中 `kdims`、`vdims`、`group`、`label` 这 4 个特别重要。

In [6]:
c4.param.values()

{'cdims': {},
 'datatype': ['dataframe',
  'dictionary',
  'grid',
  'xarray',
  'multitabular',
  'spatialpandas',
  'dask_spatialpandas',
  'dask',
  'cuDF',
  'array',
  'narwhals',
  'ibis'],
 'extents': (None, None, None, None),
 'group': 'Curve',
 'kdims': [Dimension('x')],
 'label': 'c4',
 'name': 'Curve00278',
 'vdims': [Dimension('y')]}

:::{tip}
在 Notebook 里，格子最后的表达式只要能显示，就会用选好的库画成图；用 `display()` 也能显示。

```python
from IPython.display import display
display(c1)
```
:::

## `Dimension` 对象

维度的信息放在 `Dimension` 里，属性如下。

| 属性 | 类型 | 默认值 | 说明 |
|:--|:--|:--|:--|
| `name` | String | `'Dimension'` | 维度的名字，取成 Python 的标识符 |
| `label` | String | `None` | 显示用的名字，也可以用 LaTeX |
| `cyclic` | Boolean | `False` | 值是否周期(例如角度) |
| `value_format` | Callable | `None` | 把值变成格式的函数 |
| `range` | Tuple | `(None, None)` | 值的范围 |
| `soft_range` | Tuple | `(None, None)` | 参考的范围 |
| `type` | Parameter | `None` | 值的类型 |
| `default` | Parameter | `None` | 默认值 |
| `step` | Number | `None` | 滑块的步长 |
| `unit` | String | `None` | 单位 |
| `values` | List | `[]` | 可选的值的列表 |

`Element` 有键维度的列表 `kdims` 与值维度的列表 `vdims`，两者都是 `Dimension`；种类不同，把这些维度画成图的方式也不同。例如 `Curve` 的键维度 `"x"` 是各点的 x 坐标、值维度 `"y"` 是 y 坐标。

In [7]:
print(hv.Curve.kdims, hv.Curve.vdims)

[Dimension('x')] [Dimension('y')]


把线段汇总起来的 `Segments` 有 4 个键维度，对应两端的 x 与 y，没有值维度。

In [8]:
print(hv.Segments.kdims, hv.Segments.vdims)

[Dimension('x0'), Dimension('y0'), Dimension('x1'), Dimension('y1')] []


箱线图的 `BoxWhisker` 没有键维度，值维度只有 1 个。

In [9]:
print(hv.BoxWhisker.kdims, hv.BoxWhisker.vdims)

[] [Dimension('y')]


做 `Element` 时可以用 `kdims` 与 `vdims` 决定维度，下面是 3 种写法：`c5` 用字符串，等于同时决定 `Dimension` 的 `name` 与 `label`；`c6` 用两个元组，分别决定 `name` 与 `label`；`c7` 直接传 `Dimension`，所以能用关键字决定全部属性。传的不是列表时，等于传了长度为 1 的列表。

In [10]:
c5 = hv.Curve(tuple_data, kdims="time", vdims="position", label="c5")
c6 = hv.Curve(tuple_data, kdims=("t", "Time"), vdims=("x", "Position"), label="c6")
c7 = hv.Curve(
    tuple_data,
    label="c7",
    kdims=hv.Dimension("t", label="Time", unit="s"),
    vdims=hv.Dimension("x", label="Position", unit="mm"),
)

for c in (c5, c6, c7):
    kdim = c.kdims[0]
    vdim = c.vdims[0]
    print(f"{c.label}: {kdim.name=} {kdim.label=} {kdim.unit=} ")

c5: kdim.name='time' kdim.label='time' kdim.unit=None 
c6: kdim.name='t' kdim.label='Time' kdim.unit=None 
c7: kdim.name='t' kdim.label='Time' kdim.unit='s' 


画成图时，x 轴的题由键维度的 `label` 与 `unit` 决定、y 轴的题由值维度的决定，例如 `c7` 就是这样。

In [11]:
c7.opts(width=600, show_grid=True)

:Curve   [t]   (Position)

想改维度的信息时，直接改写属性也可以，但推荐用 `redim`：它不改原来的对象，而是返回新的对象。`redim()` 能改维度本身。

❶ `c1` 的维度是 `x` 与 `y`，用 `redim()` 的关键字来决定：给 `x` 传字符串，就会做出那个名字的 `Dimension`；给 `y` 直接传 `Dimension`。

`redim` 不是方法，而是用来改属性的对象：❷ `redim.unit()` 改 `unit`、❸ `redim.range()` 改 `range`。决定了 `range` 之后，轴的显示范围就用这个值，不再是数据的范围。

In [12]:
c8 = c1.redim(x="time", y=hv.Dimension("position", label="Position"))  # ❶
c9 = c5.redim.unit(time="s", position="mm")  # ❷
c10 = c7.redim.range(t=(0, 8), x=(-2, 2))  # ❸

print(f"{c7.kdims[0].range = }, {c10.kdims[0].range = }")

c7.kdims[0].range = (None, None), c10.kdims[0].range = (0, 8)

## 从 `Dataset` 到图

数据本来就整齐的话，一个个做 `Element`、再决定维度很麻烦；从整齐数据(tidy data)出发，做图要简单得多。

二维的表里列是维度、行是观测，列分成自己决定的独立变量与测量得到的从属变量。下面的程序读入 `gapminder_CJU.csv` 得到 `df`，其中 `country` 是国家、`year` 是年份，`gdp_capita`、`life`、`population`、`ppp` 等是值。

In [13]:
import pandas as pd

df = pd.read_csv("data/gapminder_CJU.csv")
df.sample(5).sort_index()

,country,year,gdp_capita,life,ppp_capita,population
14,China,2004,2470.0,73.7,4430.0,1320000000
29,Japan,1990,38100.0,79.3,19600.0,125000000
31,Japan,1992,39500.0,79.5,21500.0,125000000
33,Japan,1994,39400.0,80.1,22400.0,126000000
78,United States,2010,48500.0,78.7,48500.0,309000000


从 `df` 做出 `Dataset`，把 `country` 与 `year` 当作键维度，剩下的列则成为值维度；显示时键维度放在方括号里、值维度放在圆括号里。

In [14]:
ds = hv.Dataset(df, kdims=["country", "year"])
ds

:Dataset   [country,year]   (gdp_capita,life,ppp_capita,population)

用 `ds.to.curve()` 把 `Dataset` 变成 `Curve` 的容器，维度由 `kdims` 与 `vdims` 决定：这里没有指定 `country`，所以它成了分组的维度，结果是 `HoloMap`，键维度是 `country`。用 `print()` 看内容，就知道 `hm` 的每个值都是 `Curve`(键是 `year`、值是 `ppp_capita`)。

In [15]:
hm = ds.to.curve(kdims="year", vdims="ppp_capita")
print(hm)

:HoloMap   [country]
   :Curve   [year]   (ppp_capita)


`HoloMap` 从 `UniformNdMapping` 派生，下面的图是它的派生类：都是持有多个元素的容器，只是显示方式不同，数据的存放方式相同。

In [16]:
print_subclasses(hv.core.UniformNdMapping)

└──UniformNdMapping
   ├──NdLayout
   ├──NdOverlay
   ├──HoloMap
   │  └──DynamicMap
   └──GridSpace
      └──GridMatrix


看各个对象的 `kdims` 与 `vdims`。`HoloMap` 能用下标取元素，所以 `hm['China']` 就取到 `country` 为 `'China'` 的元素。

In [17]:
print(f"{hm.kdims = }")
print(f'{hm["China"].kdims = }')
print(f'{hm["China"].vdims = }')

hm.kdims = [Dimension('country')]
hm["China"].kdims = [Dimension('year')]
hm["China"].vdims = [Dimension('ppp_capita')]


`data` 属性就是真正的数据：`HoloMap` 用有序字典持有，`Curve` 用 `DataFrame` 持有。

In [18]:
print("HoloMap data:")
pprint.pp(hm.data)
print()
print("Curve data:")
print(hm["China"].data.head())

HoloMap data:
{('China',): :Curve   [year]   (ppp_capita),
 ('Japan',): :Curve   [year]   (ppp_capita),
 ('United States',): :Curve   [year]   (ppp_capita)}

Curve data:
  country  year  gdp_capita  life  ppp_capita  population
0   China  1990       729.0  68.7       982.0  1180000000
1   China  1991       786.0  68.8      1090.0  1190000000
2   China  1992       887.0  69.1      1260.0  1210000000
3   China  1993       998.0  69.4      1460.0  1220000000
4   China  1994      1120.0  69.7      1660.0  1230000000


直接输出 `hm`，就能像左图那样用下拉框选 `country`，选了曲线就换掉。

In [19]:
hm.opts(width=400)

:HoloMap   [country]
   :Curve   [year]   (ppp_capita)

想同时显示多条曲线时，用 `overlay()` 变成 `NdOverlay`：`NdOverlay` 把元素叠起来画，键维度的值成为图例。

In [20]:
year_ppp_plot = hm.overlay()
print(year_ppp_plot)
year_ppp_plot

:NdOverlay   [country]
   :Curve   [year]   (ppp_capita)

:NdOverlay   [country]
   :Curve   [year]   (ppp_capita)

下面的例子按国家分开，每个元素做成 `Scatter`，键维度是 `ppp_capita`、值维度是 `life` 与 `year`；`Scatter` 把键维度当 x 轴、把第 1 个值维度(`life`)当 y 轴。加上 `hover` 工具之后，鼠标放到点上还能看到没用到的 `year` 的值。

In [21]:
gdp_life_plot = ds.to.scatter(
    kdims=["ppp_capita"], vdims=["life", "year"], groupby=["country"]
).overlay()
gdp_life_plot.opts(width=400)

:NdOverlay   [country]
   :Scatter   [ppp_capita]   (life,year)

用 `HoloMap.layout()` 变成 `NdLayout`，就能把多个图并排显示。

In [22]:
hm.layout()

:NdLayout   [country]
   :Curve   [year]   (ppp_capita)

## 并排与叠加的容器

`NdLayout` 与 `NdOverlay` 是多维的容器，按并排或叠加来汇总图；此外还有不带维度的 `Layout` 与 `Overlay`。它们能用 `+` 与 `*` 运算符做出来：`+` 是并排(`Layout`)、`*` 是叠加(`Overlay`)。下面的代码把一条曲线与点叠在一起。

```python
hv.Curve((x1, y1)) * hv.Scatter((x2, y2))
```

下面 `orth_poly()` 做出曲线与它的极值：先算点的坐标，再用 `scipy.signal.find_peaks()` 求极大与极小的位置。

❶ 做出 `Curve`，加上 `group` 与 `label`：在同一个容器里靠它们区分元素，`label` 会出现在图例里。❷ 用极值的坐标做出 `Scatter`，给它另一个 `group`。❸ 做出两个显示数值的 `Labels`。

In [23]:
import numpy as np
from scipy import special
from scipy import signal


def orth_poly(f, n, x0=-1, x1=1):
    x = np.linspace(x0, x1, 100)
    y = f(n, x)

    index_hi, _ = signal.find_peaks(y)
    index_lo, _ = signal.find_peaks(-y)
    name = f.__name__.split("_")[-1]
    label = f"{name} {n}"
    curve = hv.Curve((x, y), group="curve", label=label)  # ❶

    index = np.r_[index_hi, index_lo]
    xp = x[index]
    yp = y[index]
    peak = hv.Scatter((xp, yp), group="peak", label=label)  # ❷

    xh = x[index_hi]
    yh = y[index_hi]
    texth = ["{:4.2f}".format(v) for v in yh]
    text_hi = hv.Labels((xh, yh, texth), group="text_hi", label=label)  # ❸

    xl = x[index_lo]
    yl = y[index_lo]
    textl = ["{:4.2f}".format(v) for v in yl]
    text_lo = hv.Labels((xl, yl, textl), group="text_lo", label=label)  # ❸

    return curve * peak * text_hi * text_lo  # ❹

用 `orth_poly()` 做出 4 次切比雪夫多项式的图 `curve1`：它是 `Overlay`，用 `OrderedDict` 持有 4 个元素，键由 `group` 与 `label` 组成；注意键的文字是大写开头的。

In [24]:
from pprint import pp

curve1 = orth_poly(special.eval_chebyu, 4)
print(curve1)
print()
pp(curve1.data)

:Overlay
   .Curve.Chebyu_4   :Curve   [x]   (y)
   .Peak.Chebyu_4    :Scatter   [x]   (y)
   .Text_hi.Chebyu_4 :Labels   [x,y]   (Label)
   .Text_lo.Chebyu_4 :Labels   [x,y]   (Label)

{('Curve', 'Chebyu_4'): :Curve   [x]   (y),
 ('Peak', 'Chebyu_4'): :Scatter   [x]   (y),
 ('Text_hi', 'Chebyu_4'): :Labels   [x,y]   (Label),
 ('Text_lo', 'Chebyu_4'): :Labels   [x,y]   (Label)}


像访问属性那样就能取出元素：第 1 段属性把 `group` 相同的汇总起来取，第 2 段再取其中一个元素。

In [25]:
print(curve1.Curve)
print()
print(curve1.Curve.Chebyu_4)

:Overlay
   .Chebyu_4 :Curve   [x]   (y)

:Curve   [x]   (y)


元素的个数事先不定时，就直接用 `Overlay`：下面的代码在循环里调用 `orth_poly()`，把 3 个次数的切比雪夫多项式用 `hv.Overlay()` 叠起来，得到有 12 个元素的 `Overlay`；用 `chebyc_plot.Peak` 就能只取出 `group` 为 `peak` 的元素。

In [26]:
chebyc_plot = hv.Overlay([orth_poly(special.eval_chebyu, n) for n in [2, 3, 4]])
print(chebyc_plot)
print()
print(chebyc_plot.Peak)

:Overlay
   .Curve.Chebyu_2   :Curve   [x]   (y)
   .Peak.Chebyu_2    :Scatter   [x]   (y)
   .Text_hi.Chebyu_2 :Labels   [x,y]   (Label)
   .Text_lo.Chebyu_2 :Labels   [x,y]   (Label)
   .Curve.Chebyu_3   :Curve   [x]   (y)
   .Peak.Chebyu_3    :Scatter   [x]   (y)
   .Text_hi.Chebyu_3 :Labels   [x,y]   (Label)
   .Text_lo.Chebyu_3 :Labels   [x,y]   (Label)
   .Curve.Chebyu_4   :Curve   [x]   (y)
   .Peak.Chebyu_4    :Scatter   [x]   (y)
   .Text_hi.Chebyu_4 :Labels   [x,y]   (Label)
   .Text_lo.Chebyu_4 :Labels   [x,y]   (Label)

:Overlay
   .Chebyu_2 :Scatter   [x]   (y)
   .Chebyu_3 :Scatter   [x]   (y)
   .Chebyu_4 :Scatter   [x]   (y)


用同样的办法做出 3 个次数的勒让德多项式的 `legendre_plot`，再用 `+` 把两个并排，结果是 `Layout`；因为没有加 `group` 与 `label`，键由 HoloViews 来加。

In [27]:
legendre_plot = hv.Overlay([orth_poly(special.eval_legendre, n) for n in [2, 3, 4]])
plots = chebyc_plot + legendre_plot
print(plots)

:Layout
   .Overlay.I  :Overlay
      .Curve.Chebyu_2   :Curve   [x]   (y)
      .Peak.Chebyu_2    :Scatter   [x]   (y)
      .Text_hi.Chebyu_2 :Labels   [x,y]   (Label)
      .Text_lo.Chebyu_2 :Labels   [x,y]   (Label)
      .Curve.Chebyu_3   :Curve   [x]   (y)
      .Peak.Chebyu_3    :Scatter   [x]   (y)
      .Text_hi.Chebyu_3 :Labels   [x,y]   (Label)
      .Text_lo.Chebyu_3 :Labels   [x,y]   (Label)
      .Curve.Chebyu_4   :Curve   [x]   (y)
      .Peak.Chebyu_4    :Scatter   [x]   (y)
      .Text_hi.Chebyu_4 :Labels   [x,y]   (Label)
      .Text_lo.Chebyu_4 :Labels   [x,y]   (Label)
   .Overlay.II :Overlay
      .Curve.Legendre_2   :Curve   [x]   (y)
      .Peak.Legendre_2    :Scatter   [x]   (y)
      .Text_hi.Legendre_2 :Labels   [x,y]   (Label)
      .Text_lo.Legendre_2 :Labels   [x,y]   (Label)
      .Curve.Legendre_3   :Curve   [x]   (y)
      .Peak.Legendre_3    :Scatter   [x]   (y)
      .Text_hi.Legendre_3 :Labels   [x,y]   (Label)
      .Text_lo.Legendre_3 :Labels   [x,y

用几段属性就能取出里面的元素。

In [28]:
el = plots.Overlay.II.Peak.Legendre_3
print(f"{el}, {el.group=}, {el.label=}")

:Scatter   [x]   (y), el.group='peak', el.label='legendre 3'


输出 `plots` 时用 `opts()` 决定各元素的显示；只想改某个元素时，把它的 `group` 与 `label` 当作第 1 个参数传进去，这里就把 `text_hi` 与 `text_lo` 两个 `Labels` 分开设置。

In [29]:
plots.opts(
    opts.Layout(shared_axes=False),
    opts.Scatter(size=10, marker="x"),
    opts.Curve(alpha=0.7),
    opts.Overlay(show_grid=True, width=400, legend_cols=3),
    opts.Labels(text_font_size="9pt"),
    opts.Labels("text_hi", text_baseline="bottom"),
    opts.Labels("text_lo", text_baseline="top"),
)

:Layout
   .Overlay.I  :Overlay
      .Curve.Chebyu_2   :Curve   [x]   (y)
      .Peak.Chebyu_2    :Scatter   [x]   (y)
      .Text_hi.Chebyu_2 :Labels   [x,y]   (Label)
      .Text_lo.Chebyu_2 :Labels   [x,y]   (Label)
      .Curve.Chebyu_3   :Curve   [x]   (y)
      .Peak.Chebyu_3    :Scatter   [x]   (y)
      .Text_hi.Chebyu_3 :Labels   [x,y]   (Label)
      .Text_lo.Chebyu_3 :Labels   [x,y]   (Label)
      .Curve.Chebyu_4   :Curve   [x]   (y)
      .Peak.Chebyu_4    :Scatter   [x]   (y)
      .Text_hi.Chebyu_4 :Labels   [x,y]   (Label)
      .Text_lo.Chebyu_4 :Labels   [x,y]   (Label)
   .Overlay.II :Overlay
      .Curve.Legendre_2   :Curve   [x]   (y)
      .Peak.Legendre_2    :Scatter   [x]   (y)
      .Text_hi.Legendre_2 :Labels   [x,y]   (Label)
      .Text_lo.Legendre_2 :Labels   [x,y]   (Label)
      .Curve.Legendre_3   :Curve   [x]   (y)
      .Peak.Legendre_3    :Scatter   [x]   (y)
      .Text_hi.Legendre_3 :Labels   [x,y]   (Label)
      .Text_lo.Legendre_3 :Labels   [x,y]   (Label)
      .Curve.Legendre_4   :Curve   [x]   (y)
      .Peak.Legendre_4    :Scatter   [x]   (y)
      .Text_hi.Legendre_4 :Labels   [x,y]   (Label)
      .Text_lo.Legendre_4 :Labels   [x,y]   (Label)

### 做成嵌套

`UniformNdMapping` 的派生类能嵌套，做出复杂的图。下面 `run_sim()` 用 `solve_ivp()` 解质量-弹簧-阻尼系统的微分方程，返回 `Curve`。

In [30]:
from scipy import integrate
import numpy as np

def spring_sys(t, y, m, b, k, F):
    x, v = y
    dx = v
    dv = (F - k * x - b * v) / m
    return dx, dv

def spring_sim(x0, v0, m, b, k, F, tend, n):
    t = np.linspace(0, tend, n)
    res = integrate.solve_ivp(spring_sys, (0, tend), (x0, v0), t_eval=t, args=(m, b, k, F))
    return res.t, res.y[0], res.y[1]

dim_time = hv.Dimension('t', label='time', unit='s')
dim_pos = vdims=hv.Dimension('x', label='x', unit='m')

def run_sim(x0, v0, m, b, k, F, tend, n): 
    t, x, _ = spring_sim(x0, v0, m, b, k, F, tend, n)
    return hv.Curve((t, x), kdims=dim_time, vdims=dim_pos)

用 `np.mgrid()` 做 `m`、`b`、`k`、`F` 的 4 维格子，对每一组调用 `run_sim()`，汇总成 `HoloMap` 的 `hm`：键维度有 4 个，每个值是 `Curve`(键 `t`、值 `x`)。

In [31]:
from itertools import product

hm = hv.HoloMap(kdims=[
    hv.Dimension('m', label='mass', unit='kg'),
    hv.Dimension('b', label='dumper', unit=''),
    hv.Dimension('k', label='spring', unit='N/m'),
    hv.Dimension('F', label='force', unit='N'),
])

for args in np.mgrid[0.5:2:3j, 0.1:1.0:3j, 0.5:3:3j, 0:1:3j].reshape(4, -1).T:
    hm[tuple(args)] = run_sim(1.0, 0.0, *args, tend=10, n=50)

print(hm)

:HoloMap   [m,b,k,F]
   :Curve   [t]   (x)


把 `overlay()` 与 `layout()` 接起来变成 `NdLayout`，用 `cols()` 决定列数、用 `opts()` 决定显示。`plots` 是 `m` 与 `b` 的二维 `NdLayout`，每个元素是 `F` 的 `HoloMap`，它又由 `k` 的 `NdOverlay` 组成。

In [32]:
options = [
    opts.NdOverlay(legend_position='top_left', legend_cols=3), 
    opts.Curve(show_grid=True, width=300)    
]    
    
plots = hm.overlay('k').layout(['m', 'b']).cols(3).opts(*options)
print(plots)

:NdLayout   [m,b]
   :HoloMap   [F]
      :NdOverlay   [k]
         :Curve   [t]   (x)


In [33]:
plots

:NdLayout   [m,b]
   :HoloMap   [F]
      :NdOverlay   [k]
         :Curve   [t]   (x)

显示时，`NdLayout` 把 `m` 与 `b` 并排，`NdOverlay` 把 `k` 叠在同一张图里，`HoloMap` 则把 `F` 变成滑块。

## 图的选项

HoloViews 的元素与容器只表达数据与维度的关系，本身不带颜色与线型，外观由 `opts` 决定：`opts` 的方法名就是元素的类名，用关键字决定值。下面的例子是 `Curve` 与 `Scatter` 的选项，其中 `width` 是宽度、`show_grid` 是网格、`line_width` 是线宽、`color` 是颜色；也能指定 `group` 与 `label`，`scatter_opt2` 就只对 `group` 为 `Large` 的生效。

用 `hv.help(hv.Curve)` 会列出能用的选项与说明。

In [34]:
from holoviews import opts

curve_opt = opts.Curve(width=600, show_grid=True, line_width=4, color="green")
scatter_opt = opts.Scatter(size=16, marker="x", color="red")
scatter_opt2 = opts.Scatter("Large", size=24, color="blue")

`opts()` 设置在那个对象上，`options()` 则先复制再设置，两者都能接收多个选项；用 `opts.info()` 能看到设置，加 `show_defaults=True` 连默认值一起看，用 `opts.clear()` 能清掉。

设置并不放进对象里面，而是由 HoloViews 用一张全局的对应表来保存。

下面的代码做出 `Overlay`，传进两个选项。

In [35]:
x = np.linspace(0, 4 * np.pi, 50)
y = np.sin(x)
data = x, y
data2 = x[::5], y[::5]
overlay = hv.Curve(data) * hv.Scatter(data) * hv.Scatter(data2, group="Large")
overlay.opts(curve_opt, scatter_opt, scatter_opt2)
overlay.opts.info()
print()
overlay.opts.clear()
overlay.opts.info(show_defaults=True)

:Overlay
   .Curve.I   :Curve   [x]   (y)
    | Options(color='green', line_width=4, show_grid=True, width=600)
   .Scatter.I :Scatter   [x]   (y)
    | Options(color='red', marker='x', size=16)
   .Large.I   :Scatter   [x]   (y)
    | Options(color='blue', marker='x', size=24)

:Overlay
 | Options(axiswise=False, click_policy='mute', framewise=False)
   .Curve.I   :Curve   [x]   (y)
    | Options(axiswise=False, color=Cycle(), framewise=False, line_width=2, muted_alpha=0.2)
   .Scatter.I :Scatter   [x]   (y)
    | Options(axiswise=False, cmap='kbc_r', color=Cycle(), framewise=False, muted_alpha=0.2,
    |         size=np.float64(2.449489742783178))
   .Large.I   :Scatter   [x]   (y)
    | Options(axiswise=False, cmap='kbc_r', color=Cycle(), framewise=False, muted_alpha=0.2,
    |         size=np.float64(2.449489742783178))


默认的选项用 `opts.defaults()` 决定，这一章开头也就决定了 `Scatter`、`Curve`、`NdOverlay` 的默认值。

```python
opts.defaults(
    opts.Scatter(tools=['hover'], width=500, show_grid=True),     
    opts.Curve(tools=['hover'], width=500, show_grid=True), 
    opts.NdOverlay(legend_position='left'),
)
```

用 `opts()` 还能按数据的维度决定外观：`Points` 是二维的点，默认把键维度当作坐标，用 `opts()` 可以把大小、颜色、形状连到维度上。

下面的例子把 `life` 与 `ppp_capita` 当坐标、`year` 当颜色、`population` 当大小：维度用 `dim` 包起来，再用运算符或方法计算，这里把 `population` 用 `norm()` 归一化到 0〜1、取平方根、再乘 10，最后加一个常数，让小点也看得见。

In [36]:
from holoviews import dim

points = ds.to.points(
    kdims=["life", "ppp_capita"], vdims=["year", "population"], groupby=[]
).opts(
    opts.Points(
        color=dim("year"),
        size=dim("population").norm() ** 0.5 * 10 + 2,
        colorbar=True,
        cmap="Viridis",
        width=500,
        show_grid=True,
    )
)
points

:Points   [life,ppp_capita]   (year,population)

`dim` 的表达式只表示计算的步骤，并不当场计算，而是在出图时才算，效果与调用 `apply()` 相同。

In [37]:
size_expr = dim("population").norm() ** 0.5 * 10 + 2
size = size_expr.apply(ds)
print(size_expr)
print(f"{size.min() = }, {size.max() = }")

(((dim('population').norm())**0.5)*10)+2
size.min() = np.float64(2.0), size.max() = np.float64(12.0)


## 小结

* 用 `extension('bokeh')` 选绘制用的库。
* 数据用 `Element`(`Curve`、`Scatter` 等)包起来，用 `kdims` 与 `vdims` 决定维度。
* 维度的信息在 `Dimension` 里，要改就用 `redim`(不改原来的对象)。
* 整齐数据先做成 `Dataset`，再用 `to.curve()` 等变成图。
* 容器有 `HoloMap`(滑块)、`NdLayout`(并排)、`NdOverlay`(叠加)、`Layout`(`+`)、`Overlay`(`*`)。
* 外观用 `opts` 决定，写在元素类名的方法里、用关键字传值。
* 用 `dim` 的表达式能从维度的值算出颜色与大小。